# Tilavet — Whisper Large-v3 Quran Fine-tuning

Kuran-i Kerim tilaveti icin openai/whisper-large-v3 modelini LoRA ile fine-tune eden notebook.

| | |
|---|---|
| **Model** | openai/whisper-large-v3 (1.5B param) + LoRA |
| **Dataset** | tarteel-ai/everyayah (~234K sample, 37 kari) |
| **GPU** | A100 80GB High RAM |
| **Batch** | 32 x 2 = 64 effective |
| **Steps** | ~18,281 (5 epoch) |
| **Hedef** | WER < 3% |
| **Sure** | ~12-13 saat |

**Kullanim:** Runtime > A100 High RAM > Run All > HF token gir > Bekle

In [ ]:
# ============================================================
# ENVIRONMENT
# ============================================================
import os, shutil

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/model_cache"

from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "GPU yok! Runtime > Change runtime type > A100"

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
disk = shutil.disk_usage("/content")

print(f"GPU:     {gpu}")
print(f"VRAM:    {vram:.1f} GB")
print(f"CUDA:    {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"Disk:    {disk.free / 1e9:.1f} GB bos")

In [ ]:
# ============================================================
# DEPENDENCIES
# ============================================================
!pip install -q transformers==4.47.0 peft==0.14.0 datasets==3.2.0 accelerate==1.2.1
!pip install -q evaluate jiwer soundfile librosa tensorboard huggingface_hub
!pip cache purge

import transformers, peft, datasets
print(f"transformers:  {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"datasets:      {datasets.__version__}")

In [ ]:
# ============================================================
# HUGGINGFACE LOGIN
# ============================================================
from huggingface_hub import login
login()

In [ ]:
# ============================================================
# CHECKPOINT RESUME
# ============================================================
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = "/content/drive/MyDrive/tilavet/whisper-large-v3-quran-lora"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

resume_from = get_last_checkpoint(OUTPUT_DIR)
if resume_from:
    print(f"Checkpoint bulundu: {resume_from}")
    print("Egitim kaldigi yerden devam edecek.")
else:
    print("Checkpoint yok, sifirdan baslanacak.")

In [ ]:
# ============================================================
# IMPORTS + CONFIG
# ============================================================
import random, warnings, re, gc
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch
import evaluate
from datasets import load_dataset, Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore", message=".*attention_mask.*")
warnings.filterwarnings("ignore", message=".*Trainer.tokenizer.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")
warnings.filterwarnings("ignore", message=".*FutureWarning.*")
warnings.filterwarnings("ignore", message=".*requires_grad.*")
warnings.filterwarnings("ignore", message=".*checkpoint.*")

# --- Model ---
MODEL_ID          = "openai/whisper-large-v3"

# --- LoRA ---
LORA_R            = 32
LORA_ALPHA        = 64
LORA_DROPOUT      = 0.05
TARGET_MODULES    = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]

# --- Training ---
BATCH_SIZE        = 32
GRAD_ACCUM        = 2        # effective batch = 64
LEARNING_RATE     = 3e-4
WARMUP_STEPS      = 200
EPOCHS            = 5
TOTAL_SAMPLES     = 234_000
EFFECTIVE_BATCH   = BATCH_SIZE * GRAD_ACCUM
MAX_STEPS         = (EPOCHS * TOTAL_SAMPLES) // EFFECTIVE_BATCH

# --- Eval & Save ---
EVAL_STEPS        = 2000
SAVE_STEPS        = 2000
EVAL_BATCH_SIZE   = 16
SAVE_TOTAL_LIMIT  = 3
GEN_MAX_LENGTH    = 225

# --- Data ---
MAX_DURATION      = 30.0

# --- Paths ---
OUTPUT_DIR        = "/content/drive/MyDrive/tilavet/whisper-large-v3-quran-lora"
HUB_MODEL_ID      = "baristiran/whisper-large-v3-quran-lora"

print(f"Model:      {MODEL_ID}")
print(f"LoRA:       r={LORA_R}, alpha={LORA_ALPHA}, {len(TARGET_MODULES)} module")
print(f"Batch:      {BATCH_SIZE} x {GRAD_ACCUM} = {EFFECTIVE_BATCH}")
print(f"Steps:      {MAX_STEPS:,} (~{EPOCHS} epoch)")
print(f"Eval/Save:  her {EVAL_STEPS} step")

In [ ]:
# ============================================================
# MODEL + LoRA
# ============================================================

# Processor
processor = WhisperProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.set_prefix_tokens(language="ar", task="transcribe")
print("Processor OK")

# Base model (fp32 + SDPA)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    attn_implementation="sdpa",
)
model.config.use_cache = False
model.generation_config.language = "ar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
print(f"Base model OK — GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f"LoRA OK — GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ============================================================
# DATASET
# ============================================================
print("Dataset baglantisi kuruluyor...")

train_stream = load_dataset(
    "tarteel-ai/everyayah", split="train",
    streaming=True, trust_remote_code=False,
)
print("Train: streaming mode (disk = 0 GB)")

eval_stream = load_dataset(
    "tarteel-ai/everyayah", split="test",
    streaming=True, trust_remote_code=False,
)
print("Eval: 2000 sample RAM'e yukleniyor...")
eval_raw = list(eval_stream.take(2000))
print(f"Eval: {len(eval_raw)} sample hazir")

In [ ]:
# ============================================================
# PREPROCESSING
# ============================================================
def speed_perturb(batch):
    """Hiz perturbasyonu: 0.9x, 1.0x veya 1.1x."""
    speed = random.choice([0.9, 1.0, 1.1])
    if speed != 1.0:
        audio = np.array(batch["audio"]["array"], dtype=np.float32)
        new_len = int(len(audio) / speed)
        indices = np.linspace(0, len(audio) - 1, new_len)
        batch["audio"] = {
            "array": np.interp(indices, np.arange(len(audio)), audio),
            "sampling_rate": batch["audio"]["sampling_rate"],
        }
    return batch


def prepare_sample(batch):
    """Audio -> mel spectrogram, text -> token ids."""
    audio = batch["audio"]
    features = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="np",
    )
    batch["input_features"] = features.input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


# Train pipeline (streaming)
print("Train pipeline hazirlaniyor...")
train_dataset = (
    train_stream
    .filter(lambda x: len(x["audio"]["array"]) / x["audio"]["sampling_rate"] <= MAX_DURATION)
    .map(speed_perturb)
    .map(prepare_sample, remove_columns=["audio", "text"])
    .shuffle(buffer_size=10_000, seed=42)
)
print("Train pipeline OK")

# Eval (RAM'de)
print("Eval samples isleniyor...")
eval_processed = []
skipped = 0
for i, sample in enumerate(eval_raw):
    dur = len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    if dur <= MAX_DURATION:
        p = prepare_sample(sample)
        eval_processed.append({"input_features": p["input_features"], "labels": p["labels"]})
    else:
        skipped += 1
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(eval_raw)}...")

eval_dataset = Dataset.from_list(eval_processed)
print(f"Eval: {len(eval_dataset)} sample ({skipped} atildi)")

del eval_raw, eval_processed
gc.collect()

In [ ]:
# ============================================================
# DATA COLLATOR + METRICS
# ============================================================
@dataclass
class SpeechCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


collator = SpeechCollator(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = evaluate.load("wer")


def strip_harakat(text: str) -> str:
    """Arapca harekeler (fatha, damma, kasra vs.) kaldir."""
    return re.sub(r'[\u064B-\u065F\u0670]', '', text).strip()


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_diac = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    pred_norm = [strip_harakat(p) for p in pred_str]
    label_norm = [strip_harakat(l) for l in label_str]
    wer_norm = 100 * wer_metric.compute(predictions=pred_norm, references=label_norm)

    return {"wer_diacritized": wer_diac, "wer_normalized": wer_norm}


print("Collator + Metrics OK")

In [ ]:
# ============================================================
# TRAINING
# ============================================================
class SaveLoRACallback(TrainerCallback):
    """Her checkpoint'ta sadece LoRA adapter'i kaydet."""
    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        kwargs["model"].save_pretrained(ckpt)
        full = ckpt / "pytorch_model.bin"
        if full.exists():
            full.unlink()


training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    fp16=True,
    fp16_full_eval=True,
    gradient_checkpointing=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    generation_max_length=GEN_MAX_LENGTH,
    predict_with_generate=True,
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="wer_normalized",
    greater_is_better=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[SaveLoRACallback],
)

print("="*50)
if resume_from:
    print(f"DEVAM: {resume_from}")
else:
    print("SIFIRDAN BASLIYOR")
print(f"Steps:  {MAX_STEPS:,}  |  Batch: {EFFECTIVE_BATCH}  |  LR: {LEARNING_RATE}")
print(f"GPU:    {torch.cuda.memory_allocated() / 1e9:.1f} GB kullaniliyor")
print("="*50 + "\n")

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# ============================================================
# KAYDET + TEST
# ============================================================
# Runtime restart durumunda gerekli degiskenler
import re, torch
from datasets import load_dataset

OUTPUT_DIR     = "/content/drive/MyDrive/tilavet/whisper-large-v3-quran-lora"
MODEL_ID       = "openai/whisper-large-v3"
GEN_MAX_LENGTH = 225

def strip_harakat(text: str) -> str:
    return re.sub(r'[\u064B-\u065F\u0670]', '', text).strip()

FINAL = OUTPUT_DIR + "/final"
model.save_pretrained(FINAL)
processor.save_pretrained(FINAL)
print(f"Model kaydedildi: {FINAL}")

print("\n--- Test Inference ---")
test_stream = load_dataset(
    "tarteel-ai/everyayah", split="test",
    streaming=True, trust_remote_code=False,
)

for i, sample in enumerate(test_stream.take(5)):
    audio = sample["audio"]
    inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
    inputs = inputs.to(model.device)

    with torch.no_grad():
        ids = model.generate(**inputs, max_length=GEN_MAX_LENGTH)

    pred = processor.batch_decode(ids, skip_special_tokens=True)[0]
    ref = sample["text"]
    ok = "OK" if strip_harakat(pred) == strip_harakat(ref) else "XX"

    print(f"\n[{i+1}] {ok}")
    print(f"  Ref:  {ref}")
    print(f"  Pred: {pred}")

# ============================================================
# HUGGINGFACE HUB'A YUKLE
# ============================================================
HUB_MODEL_ID = "baristiran/whisper-large-v3-quran-lora"
print(f"\nHub'a yukleniyor: {HUB_MODEL_ID}")
model.push_to_hub(HUB_MODEL_ID)
processor.push_to_hub(HUB_MODEL_ID)
print(f"Model yuklendi: https://huggingface.co/{HUB_MODEL_ID}")

print("\nTAMAM.")